# Studio 3.1 — Pyomo Basics: Hard-Coded vs. Soft-Coded Models

**OPIM 5641 · Business Decision Modeling · Dr. Dave Wanik, UConn**

So far we've solved LPs three ways: brute force, the graphical method, and Simplex by hand. Tonight
we hand the work to a solver. You write down the model, **Pyomo** builds it, and the solver does the
Simplex for you.

We start with a problem you already know the answer to: **Flair's Furniture (320 tables, 360 chairs,
\$4,040)**. If the solver agrees, the code is right.

We will write it two ways:

- **Version A, hard-coded:** every number from the word problem is typed straight into the model.
- **Version B, soft-coded:** the numbers live in dictionaries, and the model reads them with loops.

**How this notebook works.** All of the code is already here. You do not have to write a model
from scratch. Your job is to **run it, read it, and change it**: each experiment has a clearly marked
cell where you plug in a scenario, re-run, and see what happens.

## Import modules (download Pyomo)

Run this first, every time. It installs Pyomo and the `cbc` solver in Colab.

In [ ]:
# import modules

# this makes beautiful plots using pylab (matplotlib)
%matplotlib inline
from pylab import * # * means import ALL NAME SPACES
import pandas as pd

# useful modules for downloading pyomo onto Colab
import shutil
import sys
import os.path

# install pyomo if it doesn't exist
if not shutil.which("pyomo"):
    !pip install -q pyomo
    assert(shutil.which("pyomo"))

# install the 'cbc' solver if it doesn't exist
if not (shutil.which("cbc") or os.path.isfile("cbc")):
    if "google.colab" in sys.modules:
        !apt-get install -y -qq coinor-cbc
    else:
        try:
            !conda install -c conda-forge coincbc
        except:
            pass

# make sure 'cbc' is the solver that we will invoke later on
assert(shutil.which("cbc") or os.path.isfile("cbc"))

# import ALL NAMESPACES (variables) from pyomo
from pyomo.environ import *

## The problem

Flair makes **tables** and **chairs**.

| | Tables | Chairs | Hours available |
|---|---|---|---|
| Carpentry | 3 | 4 | 2,400 |
| Painting | 2 | 1 | 1,000 |
| **Profit (USD)** | 7 | 5 | - |

Plus: **no more than 450 chairs**, and **at least 100 tables**.

$$Max\ Z = 7T + 5C$$

subject to:
* $3T + 4C \le 2400$ `carpentry`
* $2T + 1C \le 1000$ `painting`
* $C \le 450$ `chair limit`
* $T \ge 100$ `table minimum`
* $T, C \ge 0$ `non-negativity`

# Version A · The hard-coded model

Every Pyomo model has the same pieces, in the same order:

**model → variables → objective → constraints → solve → read the results**

Read the cell below next to the math above. Each line of math became one line of code.

In [ ]:
# you will always need to write this code
model = ConcreteModel()

# declare decision variables
model.tables = Var(domain=NonNegativeReals)
model.chairs = Var(domain=NonNegativeReals)

# declare objective
model.profit = Objective(
                      expr = 7*model.tables + 5*model.chairs, # values come from the table
                      sense = maximize)

# write the constraints one by one, don't forget LHS vs. RHS
# it's OK to have mixed constraints (LHS <= RHS vs. LHS >= RHS)!!!
model.constraint1 = Constraint(expr = 3*model.tables + 4*model.chairs <= 2400) # carpentry hours
model.constraint2 = Constraint(expr = 2*model.tables + 1*model.chairs <= 1000) # painting hours
model.constraint3 = Constraint(expr = model.chairs <= 450) # chair limit
model.constraint4 = Constraint(expr = model.tables >= 100) # table minimum

# show the model you've created
model.pprint()

In [ ]:
#            solver          path the solver
SolverFactory('cbc', executable='/usr/bin/cbc').solve(model).write()

Look for `Termination condition: optimal`. If you see it, you solved! If not, you probably have a
bug (a typo in a constraint, the wrong sense, etc.).

Now read the results. **Don't forget the `()` after everything** - lots of students forget this and
get a bug!

In [ ]:
print("Profit = ", model.profit(), " USD")
print("Tables = ", model.tables(), " units")
print("Chairs = ", model.chairs(), " units")
print()

# the left-hand side of each constraint - how much did we actually use?
print("Carpentry = ", model.constraint1(), " hours (of 2400)")
print("Painting = ", model.constraint2(), " hours (of 1000)")
print("Chairs = ", model.constraint3(), " units (limit 450)")
print("Tables = ", model.constraint4(), " units (minimum 100)")
print()

# slack = what is left over (0 means it's all used up!)
print("Carpentry slack = ", model.constraint1.uslack(), " hours")
print("Painting slack = ", model.constraint2.uslack(), " hours")
print("Chair slack = ", model.constraint3.uslack(), " units")

**320 tables, 360 chairs, \$4,040.** Same answer as brute force, the graphical method, and
Simplex. Fourth way, same answer.

Carpentry and painting are **binding** (no slack): every hour is used up. Those are the bottlenecks.
The chair limit has 90 units of slack, the same 90 you found on your Simplex math check.

### Try it: Scenario A1

The shop adds a shift, so carpentry goes from **2,400 hours to 3,000 hours**.

1. Go up to the hard-coded model, find the `2400`, and change it to `3000`.
2. Re-run the model cell, the solve cell and the results cell.
3. Write down the new profit: \_\_\_\_\_\_\_\_
4. **Change it back to 2400** and re-run.

That was easy with one number. Now imagine 40 constraints, and the same number buried in six of them.
Or imagine the boss asks you to add a third product: you would have to edit the variables, the
objective, and **every** constraint. That is the problem with hard-coding.

# Version B · The soft-coded model

Same problem. This time the numbers go in **dictionaries**, and the model reads them with **loops**.

Three cells, always run **in this order**:

1. **THE DATA** - the word problem, as dictionaries. Running it resets everything to the original problem
   (it does not touch your scoreboard).
2. **THE SCENARIO** - where you change things.
3. **THE MODEL** - builds the model from whatever is in the dictionaries, solves it, prints it. **You
   never edit this cell.**

In [ ]:
# ---------- 1. THE DATA (run this to reset to the original problem) ----------
products = ["tables", "chairs"]

# profit per unit
profit = {
    "tables" : 7,
    "chairs" : 5
}

# hours each product needs in each department
hours = {
    "carpentry" : {"tables" : 3, "chairs" : 4},
    "painting"  : {"tables" : 2, "chairs" : 1}
}

# hours available this month
total_time = {
    "carpentry" : 2400,
    "painting"  : 1000
}

# marketing rules
max_units = {"chairs" : 450}    # no more than 450 chairs
min_units = {"tables" : 100}    # at least 100 tables

**The scoreboard.** One more dictionary, and it starts out empty. Every time you run THE MODEL cell,
its last line adds one entry:

`scoreboard[scenario] = model.profit()`

That is how you add to a dictionary: there is no `append`. You assign a value to a **new key**, and the
dictionary grows by one. (Assign to a key that is already there, and it overwrites the old value.)

Run this cell **once**. If you run it again, you wipe the scoreboard clean.

In [ ]:
# an empty scoreboard: scenario name -> profit
scoreboard = {}

In [ ]:
# ---------- 2. THE SCENARIO (change things here) ----------
scenario = "base"

# Scenario B1: the shop adds a shift (remove the # from the next two lines)
# scenario = "B1 more carpentry"
# total_time["carpentry"] = 3000

# Scenario B2: a new product, desks (remove the # from the next five lines)
# scenario = "B2 add desks"
# products.append("desks")
# profit["desks"] = 10
# hours["carpentry"]["desks"] = 4
# hours["painting"]["desks"] = 2

# Scenario B3: your own idea goes here


In [ ]:
# ---------- 3. THE MODEL (you never change this cell) ----------
model = ConcreteModel()

# ONE indexed variable: model.x["tables"], model.x["chairs"], ...
model.x = Var(products, domain=NonNegativeReals)

# objective: add up profit x units, one product at a time
obj_expr = 0
for product in products:
    obj_expr += profit[product] * model.x[product]

model.profit = Objective(
                      expr = obj_expr,
                      sense = maximize)

# constraints -> we populate the list with loops
model.constraints = ConstraintList()

# one constraint per department
for dept in total_time:
    dept_expr = 0
    for product in products:
        dept_expr += hours[dept][product] * model.x[product]
    model.constraints.add(dept_expr <= total_time[dept])

# the marketing rules
for product in max_units:
    model.constraints.add(model.x[product] <= max_units[product])
for product in min_units:
    model.constraints.add(model.x[product] >= min_units[product])

# solve it
SolverFactory('cbc', executable='/usr/bin/cbc').solve(model)

# read the results
print("SCENARIO:", scenario)
print("Profit = ", model.profit(), " USD")
for product in products:
    print(product, " = ", model.x[product](), " units")

# the constraint list is numbered 1, 2, 3 ... in the order we added them
k = 1
for dept in total_time:
    print(dept, "hours used = ", model.constraints[k](), " of ", total_time[dept])
    k += 1

# add this scenario to the scoreboard dictionary: scoreboard[key] = value
scoreboard[scenario] = model.profit()
print()
print("Scoreboard so far:", scoreboard)

Same answer: **320 tables, 360 chairs, \$4,040.**

But look at the model cell: there is **not a single number in it.** No 2400, no 7, no 450. It does not
even know how many products there are. Everything comes from the data.

### Try it: Scenarios B1 and B2

For each scenario: run **THE DATA** cell (to reset), remove the `#` from that scenario's lines in
**THE SCENARIO** cell, run it, then run **THE MODEL** cell. Put the `#` back when you are done.

| Scenario | What changes | Profit | What did the plan look like? |
|---|---|---|---|
| base | nothing | \$4,040 | 320 tables, 360 chairs |
| B1 | carpentry hours go to 3,000 | | |
| B2 | add desks (profit \$10, carpentry 4 hours, painting 2 hours) | | |
| B3 | your own idea | | |

Notice what you did for B2: you added a whole new product by adding **four lines of data**. You did
not touch the model. Try doing that to Version A!

In [ ]:
# the scoreboard so far
print(scoreboard)

## Hard-coded vs. soft-coded

| | Hard-coded (Version A) | Soft-coded (Version B) |
|---|---|---|
| Where the numbers live | inside the model | in dictionaries, above the model |
| Easy to read next to the math? | **yes** - one line of math, one line of code | takes a minute to see the loops |
| Change one number | hunt for it in the model | change it in one place |
| Add a product or a department | rewrite the variables, objective and constraints | add a few lines of data |
| 77 products instead of 2 | no thanks | same code |

**Both are fine.** Hard-code when you are learning or the model is tiny. Soft-code when the model is
big, when the data will change, or when someone else has to use your work. In the next notebook the
model has 77 decision variables, and the code is no longer than this one.

## Bottom line

- The Pyomo recipe never changes: **model → variables → objective → constraints → solve → read the
  results.**
- A **ConstraintList** plus a loop lets you write a constraint once and use it for every department.
- **Soft-code the data** and the model does not change when the data does.

**Before you move on:** File → Save a copy in GitHub.